# Use Case 3: AI Agent Long-Term Memory System

**The Concept:** 
Basic chatbots answer individual questions, but autonomous AI Agents need to remember historical facts, user preferences, and changing states across thousands of conversations.

**The Architecture:** 
SochDB provides a dedicated **Memory System** (`sochdb.memory`). It uses an **Extraction Pipeline** to convert messy LLM text into structured facts (Entities, Relations, Assertions), and a **Consolidator** to resolve contradicting facts over time using event-sourcing (Time-Travel).

---

### Step 0: Install Packages & Configure the AI Models
We load Azure configuration using `.env`. Ensure your file is located in the same directory as this notebook.

In [1]:
!pip install sochdb openai python-dotenv

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

CHAT_MODEL = "gemini-3-flash-preview"
EMBEDDING_MODEL = "gemini-embedding-001"

def extract_json_from_llm(system_prompt, user_text):
    """Calls Gemini requesting structured JSON output (which natively supports the OpenAI Python SDK format)."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_text}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)


You should consider upgrading via the '/Users/sushanth/sochdb_python/venv/bin/python3 -m pip install --upgrade pip' command.


### Step 1: Initialize Database & Schema
We open the database and define the mental model (schema) for our AI Agent.

In [2]:
from sochdb import Database
from sochdb.memory import ExtractionPipeline, ExtractionSchema

db = Database.open("./agent_memory_app_db")

# Define the entity and relationship types we care about
schema = ExtractionSchema(
    entity_types=["person", "company", "technology", "role"],
    relation_types=["works_at", "likes", "uses", "title_is"],
    min_confidence=0.7
)

pipeline = ExtractionPipeline.from_database(db, namespace="agent_user_1", schema=schema)

### Step 2: Extract Facts from Conversations (Using Azure GPT-4)
When the user speaks to the Agent, the agent invokes the real GPT-4 model, passes it the extraction prompt, and automatically parses out the structured dictionaries.

In [6]:
# Give the LLM a rigid prompt so its Output format matches the `mock_extractor` dictionary.
LLM_EXTRACTION_PROMPT = """
You are a Knowledge Extraction Engine. Convert the user's text into an exact JSON dictionary matching this structure:
{
  "entities": [ {"name": "string", "entity_type": "person|company|technology|role"} ],
  "relations": [ {"from_entity": "string", "relation_type": "works_at|likes|uses|title_is", "to_entity": "string"} ],
  "assertions": []
}
Return only raw, valid JSON.
"""

def real_llm_extractor(text):
    print("Calling Gemini to extract schema...")
    return extract_json_from_llm(LLM_EXTRACTION_PROMPT, text)

# The pipeline extracts facts from the chat log using Gemini-3-Flash
text_input = "Hey there, I am a software engineer named Alice. I just got hired at Acme Corp and I absolutely love building things with SochDB."
result = pipeline.extract_and_commit(
    text=text_input, 
    extractor=real_llm_extractor
)

print(f"\n✅ Agent successfully dynamically memorized: {len(result.entities)} entities and {len(result.relations)} relations.")
print("Relations parsed by Gemini-3-Flash:")
for r in result.relations:
    print(f"  - [{r.from_entity}] -> {r.relation_type} -> [{r.to_entity}]")

Calling Gemini to extract schema...

✅ Agent successfully dynamically memorized: 4 entities and 4 relations.
Relations parsed by Gemini-3-Flash:
  - [Alice] -> title_is -> [software engineer]
  - [Alice] -> works_at -> [Acme Corp]
  - [Alice] -> uses -> [SochDB]
  - [Alice] -> likes -> [SochDB]


### Step 3: Consolidating Contradictions
If Alice later says she moved to Google, the Consolidator updates her canonical state while keeping a historical record of her previous job.

In [9]:
from sochdb.memory import Consolidator, RawAssertion, ConsolidationConfig

consolidator = Consolidator.from_database(db, namespace="agent_user_1")

# Convert extracted relations into RawAssertions for the consolidator
for r in result.relations:
    assertion = RawAssertion(
        id="",  # auto-generated
        fact={"subject": r.from_entity, "predicate": r.relation_type, "object": r.to_entity},
    )
    consolidator.add(assertion)

# The agent runs consolidation to clean and merge its memory banks
updated_count = consolidator.consolidate()
print(f"Consolidated {updated_count} facts.")

# The agent can now look up the absolute "Truth" about Alice before responding to queries
facts = consolidator.get_canonical_facts()
print("\nCanonical Facts in Memory (SochDB Graph):")
for fact in facts:
    f = fact.merged_fact
    print(f"- [{f['subject']}] {f['predicate']} [{f['object']}]")

Consolidated 0 facts.

Canonical Facts in Memory (SochDB Graph):
